# 02 - Review summary table

- First look at `STEAM_DB.STEAM_RAW.REVIEW_SUMMARY`. 
- dlt loads it from the `GetAppReviews` endpoint, which works without an API key.

- One row is one game on one run, with Steam's total review counts for that game. We don't load single reviews, because a game like Stellaris would need thousands of requests (see plan 06 in scratch/docs).

- The data is one load from 2026-10-26 (load id `1793014149.3994608`), 81 Paradox games. 
- The notebook only runs SELECT and DESCRIBE, so it changes nothing in Snowflake.

- The text under each query is written by hand after looking at the result. Where it says "not run yet", that cell has not been run in the notebook.

- Limits: there is only one snapshot, so no history yet. Steam gives no sales numbers, so reviews are only a stand-in for how a game is received, and we can show correlation, not causation.

## 0. Connect to Snowflake

Same connection as notebook 01. It reads the settings from `.env`, logs in as `STEAM_LOADER` and prints no secrets. Times are shown in UTC. Pick the project's `.venv` as the kernel.

In [14]:
import os
from pathlib import Path

import pandas as pd
import snowflake.connector

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [ ]:
def find_repo_root() -> Path:
    """Walk up from the notebook's folder until we find pyproject.toml."""
    here = Path.cwd().resolve()
    return next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())


def load_env_file(path: Path) -> None:
    """Copy KEY=value lines from a .env file into os.environ (never prints values)."""
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


load_env_file(find_repo_root() / ".env")

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_LOADER_USER"],
    private_key_file=os.environ["SNOWFLAKE_LOADER_PRIVATE_KEY_PATH"],
    role=os.environ["SNOWFLAKE_LOADER_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="STEAM_RAW",
    session_parameters={"TIMEZONE": "UTC"},
)


def query(sql: str) -> pd.DataFrame:
    """Run one read-only statement and return the result as a DataFrame."""
    with conn.cursor() as cursor:
        cursor.execute(sql)
        columns = [column[0] for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)

In [16]:
# Sanity check: we should be the loader role, in STEAM_DB.STEAM_RAW.
query("""
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema
""")

,ROLE,WAREHOUSE,DATABASE,SCHEMA
0,STEAM_LOADER,STEAM_WH,STEAM_DB,STEAM_RAW


## 1. Rows per load

- Every run adds one row per game, so each run has its own `collected_at`.

**💡Insights:** 
- one line with 81 rows, 81 different games and one `collected_at`. 

In [17]:
query("""
SELECT collected_at, COUNT(*) AS rows_loaded, COUNT(DISTINCT steam_appid) AS distinct_games
FROM review_summary
GROUP BY collected_at
ORDER BY collected_at
""")

,COLLECTED_AT,ROWS_LOADED,DISTINCT_GAMES
0,2026-10-26 11:29:09.930573+00:00,81,81


## 2. Duplicates

A game should appear only once per run. This lists any game that shows up twice in the same run.

**💡Insights:**
- No rows came back (empty result), so no game appears twice in the same run.
- This agrees with section 1: 81 rows for 81 games.

In [18]:
query("""
SELECT steam_appid, collected_at, COUNT(*) AS copies
FROM review_summary
GROUP BY steam_appid, collected_at
HAVING COUNT(*) > 1
""")

,STEAM_APPID,COLLECTED_AT,COPIES


## 3. Column types

**💡Insights:**
- `COLLECTED_AT` is `TIMESTAMP_LTZ(9)`, the same type as in the `games` table.
- `STEAM_APPID`, `REVIEW_SCORE`, `TOTAL_POSITIVE`, `TOTAL_NEGATIVE` and `TOTAL_REVIEWS` are `NUMBER(19,0)`.
- `REVIEW_SCORE_DESC` is `VARCHAR`.
- dlt added `_DLT_LOAD_ID` and `_DLT_ID`. Every other column can be NULL.

In [19]:
query("DESCRIBE TABLE review_summary")[["name", "type", "null?"]]

,name,type,null?
0,COLLECTED_AT,TIMESTAMP_LTZ(9),Y
1,STEAM_APPID,"NUMBER(19,0)",Y
2,REVIEW_SCORE,"NUMBER(19,0)",Y
3,REVIEW_SCORE_DESC,VARCHAR(16777216),Y
4,TOTAL_POSITIVE,"NUMBER(19,0)",Y
5,TOTAL_NEGATIVE,"NUMBER(19,0)",Y
6,TOTAL_REVIEWS,"NUMBER(19,0)",Y
7,_DLT_LOAD_ID,VARCHAR(16777216),N
8,_DLT_ID,VARCHAR(16777216),N


## 4. Nulls and consistency

Positive plus negative should equal the total, and nothing should be NULL.

**💡Insights:** 
- 5 games with no reviews, 0 rows where the numbers don't add up and 0 rows with NULL (checked in Snowsight on 2026-10-26).

In [20]:
query("""
SELECT
    COUNT(*) AS games,
    COUNT_IF(total_reviews = 0) AS games_without_reviews,
    COUNT_IF(total_positive + total_negative <> total_reviews) AS sum_mismatch,
    COUNT_IF(review_score IS NULL OR total_reviews IS NULL) AS null_rows
FROM review_summary
""")

,GAMES,GAMES_WITHOUT_REVIEWS,SUM_MISMATCH,NULL_ROWS
0,81,5,0,0


## 5. Games without reviews

- Notebook 01 found 4 unreleased games, but 5 games have no reviews. This shows them with their release info.

**💡Insights:** 
- Four are the unreleased games: Afterworld, LEGO Skylines, Guardians of the Wild Sky and Prison Architect 2. All four have `coming_soon = True`.
- The fifth is Knights of Pen and Paper 2: Free Edition (`794820`). It is released (21 Feb, 2018), free (`is_free = True`), `type = game` and not coming soon. So a released game can also have zero reviews.
- Steam answered normally for all five (zeros and the label `No user reviews`), so it is not an error.
- We don't know why the Free Edition has no reviews. Maybe its reviews sit under the paid game (Knights of Pen and Paper 2 has 1,147), but that is only a guess and we have not checked it.
- For dbt: zero reviews does not mean unreleased. Use `coming_soon` from the `games` table for that.

In [21]:
query("""
SELECT g.steam_appid, g.name, g.type, g.release_date_text, g.coming_soon, g.is_free
FROM review_summary r
JOIN (
    SELECT *
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = r.steam_appid
WHERE r.total_reviews = 0
ORDER BY g.coming_soon DESC, g.name
""")

,STEAM_APPID,NAME,TYPE,RELEASE_DATE_TEXT,COMING_SOON,IS_FREE
0,4920230,Afterworld,game,To be announced,True,False
1,1579420,Guardians of the Wild Sky,game,Coming soon,True,False
2,1573550,LEGO® Skylines,game,Coming soon,True,False
3,1257610,Prison Architect 2,game,To be announced,True,False
4,794820,Knights of Pen and Paper 2: Free Edition,game,"21 Feb, 2018",False,True


## 6. Sentiment per game

Percent positive is `total_positive / total_reviews`. It is calculated here only to read it; dbt will do it later.

**💡Insights:**

**Most reviews**

| Game | Reviews | % positive |
|---|---:|---:|
| Hearts of Iron IV | 123,314 | 91.0 % |
| Cities: Skylines | 100,557 | 92.8 % |
| Stellaris | 76,804 | 86.1 % |

**Weaker big games**

| Game | Reviews | % positive | Label |
|---|---:|---:|---|
| Cities: Skylines II | 34,961 | 58.1 % | Mixed |
| Vampire: The Masquerade - Bloodlines 2 | 6,682 | 55.7 % | Mixed |
| Star Trek: Infinite | 2,350 | 39.4 % | Mostly Negative |

**Range among games with at least 10 reviews**

| | Game | Reviews | % positive |
|---|---|---:|---:|
| Lowest | King Arthur: Fallen Champions | 27 | 37.0 % |
| Highest | Majesty Gold HD | 374 | 93.9 % |

**Edge cases**

| Case | What the data shows | Why it matters |
|---|---|---|
| Hearts of Iron III: Semper Fi | 1 review, 100 % positive, label `1 user reviews` | The label is not a fixed list of words, and one review says nothing. |
| 5 games with no reviews | Percent is NULL | `NULLIF` avoids dividing by zero. |

**Labels in the data**

| Label | Present |
|---|:---:|
| Very Positive | ✅ |
| Mostly Positive | ✅ |
| Mixed | ✅ |
| Mostly Negative | ✅ |
| `No user reviews` | ✅ |
| `1 user reviews` | ✅ |
| Overwhelmingly Positive | ❌ |
| Overwhelmingly Negative | ❌ |

In [22]:
query("""
SELECT g.name, r.review_score_desc, r.total_reviews,
       ROUND(100 * r.total_positive / NULLIF(r.total_reviews, 0), 1) AS pct_positive
FROM review_summary r
JOIN (
    SELECT steam_appid, name
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = r.steam_appid
ORDER BY r.total_reviews DESC
""")

,NAME,REVIEW_SCORE_DESC,TOTAL_REVIEWS,PCT_POSITIVE
0,Hearts of Iron IV,Very Positive,123314,91.0
1,Cities: Skylines,Very Positive,100557,92.8
2,Stellaris,Very Positive,76804,86.1
3,Crusader Kings III,Very Positive,50404,90.4
4,Europa Universalis IV,Very Positive,47752,87.0
5,Cities: Skylines II,Mixed,34961,58.1
6,Prison Architect,Very Positive,34190,88.5
7,Crusader Kings II,Very Positive,22397,92.7
8,Victoria 3,Mostly Positive,20706,70.1
9,BATTLETECH,Very Positive,18342,85.5


## 7. How many reviews do games have?

A percent from 12 reviews is much less reliable than one from 120,000, so this counts the games by number of reviews.

**💡Insights:** 

| Reviews | Games |
|---|---:|
| none | 5 |
| 1-99 | 8 |
| 100-999 | 30 |
| 1,000-9,999 | 25 |
| 10,000 or more | 13 |

- The query matches the hand count from section 6 (81 games in total).
- 43 of 81 games have fewer than 1,000 reviews, so their percent positive is noisy. Only 13 games have 10,000 or more.

In [23]:
query("""
SELECT
    CASE
        WHEN total_reviews = 0 THEN '0 none'
        WHEN total_reviews < 100 THEN '1-99'
        WHEN total_reviews < 1000 THEN '100-999'
        WHEN total_reviews < 10000 THEN '1,000-9,999'
        ELSE '10,000+'
    END AS review_volume,
    COUNT(*) AS games
FROM review_summary
GROUP BY review_volume
ORDER BY MIN(total_reviews)
""")

,REVIEW_VOLUME,GAMES
0,0 none,5
1,1-99,8
2,100-999,30
3,"1,000-9,999",25
4,"10,000+",13


## 8. Does Steam's label match the percent?

For each label, the lowest and highest percent positive.

**💡Insights:** 

| Score | Label | Games | Lowest % | Highest % |
|---:|---|---:|---:|---:|
| 0 | `1 user reviews` | 1 | 100.0 | 100.0 |
| 0 | `No user reviews` | 5 | NULL | NULL |
| 4 | Mostly Negative | 3 | 37.0 | 39.4 |
| 5 | Mixed | 27 | 40.3 | 69.6 |
| 6 | Mostly Positive | 11 | 70.1 | 79.8 |
| 8 | Very Positive | 34 | 80.6 | 93.9 |

- In this data the bands do not overlap: about 40 % separates Mostly Negative from Mixed, 70 % separates Mixed from Mostly Positive, and 80 % separates Mostly Positive from Very Positive. This is what we see in 81 games, not a rule published by Steam.
- `review_score = 0` is used for both "no reviews" and "1 user reviews". A score of 0 does not mean a bad game.
- There is no score 7 or 9 and no Overwhelmingly label in the data.

In [24]:
query("""
SELECT review_score, review_score_desc, COUNT(*) AS games,
       MIN(ROUND(100 * total_positive / NULLIF(total_reviews, 0), 1)) AS min_pct,
       MAX(ROUND(100 * total_positive / NULLIF(total_reviews, 0), 1)) AS max_pct
FROM review_summary
GROUP BY review_score, review_score_desc
ORDER BY review_score
""")

,REVIEW_SCORE,REVIEW_SCORE_DESC,GAMES,MIN_PCT,MAX_PCT
0,0,1 user reviews,1,100.0,100.0
1,0,No user reviews,5,None,None
2,4,Mostly Negative,3,37.0,39.4
3,5,Mixed,27,40.3,69.6
4,6,Mostly Positive,11,70.1,79.8
5,8,Very Positive,34,80.6,93.9


## 9. First look: price and sentiment

- A first look at one of the business questions, using the latest `games` snapshot. 
- Only games that have a price and at least 100 reviews. One snapshot of 81 games can't show cause and effect.

**💡Insights:**
- 65 games qualify (they have a price and at least 100 reviews).
- The list is sorted by percent positive. In the lowest 20 (38.4 % to 65.6 %), prices go from 9.99 to 59.99 EUR, so cheap and expensive games both appear.
- Expensive games sit at both ends: Hearts of Iron IV (49.99 EUR) is at 91.0 %, Vampire: The Masquerade - Bloodlines 2 (59.99 EUR) is at 55.7 %. We have not calculated a correlation, and one snapshot would not prove anything.
- `price_final` is the price after discount. 5 games were discounted at the time of the snapshot: Empire of Sin (75 %), Imperator: Rome (80 %), Victoria 3 (70 %), Across the Obelisk (50 %) and Shadowrun Returns (80 %). For a fair price comparison dbt should also keep `price_initial`.
- `price_final / 100` shows as `19.990000`. In staging we cast it to 2 decimals.

In [25]:
query("""
SELECT g.name, g.price_final / 100 AS price_eur, g.discount_percent,
       r.total_reviews,
       ROUND(100 * r.total_positive / NULLIF(r.total_reviews, 0), 1) AS pct_positive
FROM review_summary r
JOIN (
    SELECT *
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = r.steam_appid
WHERE g.price_final IS NOT NULL AND r.total_reviews >= 100
ORDER BY pct_positive
""")

,NAME,PRICE_EUR,DISCOUNT_PERCENT,TOTAL_REVIEWS,PCT_POSITIVE
0,Sword of the Stars II: Enhanced Edition,19.990000,0,534,38.4
1,Star Trek: Infinite,29.990000,0,2350,39.4
2,Ancient Space,19.990000,0,207,43.5
3,Impire,19.990000,0,259,44.4
4,Empire of Sin,9.990000,75,2625,51.7
5,Cities in Motion 2,19.990000,0,580,54.1
6,Vampire: The Masquerade® - Bloodlines™ 2,59.990000,0,6682,55.7
7,King Arthur II: The Role-Playing Wargame,9.990000,0,222,56.3
8,Crusader Kings Complete,9.990000,0,112,56.3
9,Surviving Mars: Relaunched,39.990000,0,1549,57.4


## 10. Findings and decisions for dbt staging

Each finding says what staging should do. These are proposals until we build `feat/dbt-staging`.

1. **Grain.** One row per `steam_appid` per `collected_at`, with no duplicates (section 2).
   - *Staging:* add a `unique` test on the pair (`steam_appid`, `collected_at`), `not_null` on both, and a relationship test to `games`.

2. **Time.** `collected_at` is `TIMESTAMP_LTZ(9)`, so it shows in whatever time zone the session uses (section 3).
   - *Staging:* convert it explicitly to UTC, and add a calendar date for joining with `games`. Both feeds run on the same day, a few seconds apart.

3. **Counts add up.** Positive + negative = total in all 81 rows, and there are no NULLs (section 4).
   - *Staging:* keep the three counts as they are, and add a test that the sum matches.
   - *Staging:* calculate `pct_positive = total_positive / NULLIF(total_reviews, 0)`. It is NULL for games with no reviews.

4. **Games with no reviews.** 5 games: 4 unreleased and one released free game, Knights of Pen and Paper 2: Free Edition (section 5).
   - *Staging:* keep these rows and add a flag such as `has_reviews`. Do not use zero reviews to mean "unreleased"; use `coming_soon` from `games`.

5. **The label is free text.** Besides the usual labels there is `1 user reviews`, and `review_score = 0` is used both for "no reviews" and for "1 user review" (sections 6 and 8).
   - *Staging:* keep the label and score as raw columns, but do not test the label with a fixed list of values, and do not use it as the only sentiment measure.
   - *Marts:* base the analysis on `pct_positive` and `total_reviews`, and show the review count next to every percent.

6. **Small samples.** 43 of 81 games have fewer than 1,000 reviews and 8 have fewer than 100 (section 7).
   - *Marts:* compare sentiment only for games above a minimum number of reviews. 100 is the proposal and is used in section 9. It is not tested yet.

7. **Totals, not new reviews.** The counts are all-time totals. A day's change is the difference between two snapshots. For example, Stellaris got one new negative review between our probe and our test run.
   - *Intermediate:* compute daily change per game with `LAG` over `collected_at`. We have not checked whether a total can go down (for example if reviews are removed), so the model should not assume it cannot.

8. **Price comparison.** `games.price_final` is the discounted price and 5 games were discounted (section 9).
   - *Intermediate:* join `review_summary` to `games` on `steam_appid` and the collection date. Keep both `price_initial` and `price_final`.

9. **History.** There is one snapshot of reviews so far. Each daily run adds 81 rows, and these repeated rows are the history, not an error.

**Still open**
- Why Knights of Pen and Paper 2: Free Edition has no reviews.
- Minimum review count for comparisons (100 is only a proposal).
- Whether the totals can decrease between snapshots (needs more than one run).
- Price against sentiment: no correlation calculated yet, and it needs more snapshots.

In [26]:
# conn.close()